# 06 — Robot Robustness Analysis

Bu notebook iki stratejiyi karşılaştırır:

- **Baseline:** 2.0 ATR initial stop, 2.5 ATR trailing, %6 activation
- **Candidate:** 1.5 ATR initial stop, 2.5 ATR trailing, %6 activation

Amaç yeni bir optimum bulmak değil; Candidate sonucunun:

- dönemler arasında,
- daha yüksek işlem maliyetlerinde,
- yakın parametre değerlerinde

dayanıklı kalıp kalmadığını kontrol etmektir.


In [ ]:
from pathlib import Path
from dataclasses import replace
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.config import StrategyConfig, PortfolioConfig
from src.features import add_indicators
from src.signals import build_market_regime
from src.experiments import (
    evaluate_strategy,
    run_strategy_grid,
)
from src.metrics import yearly_performance

sns.set_theme(style="whitegrid")


## 1. Verileri ve özellikleri hazırla


In [ ]:
stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)
market_regime = build_market_regime(market_features)

LAST_DATE = stock_features["Date"].max().strftime("%Y-%m-%d")

PERIODS = {
    "Development": ("2018-01-01", "2022-12-31"),
    "Validation": ("2023-01-01", "2024-12-31"),
    "Holdout": ("2025-01-01", LAST_DATE),
    "Full": ("2018-01-01", LAST_DATE),
}

base_portfolio = PortfolioConfig()

baseline_strategy = StrategyConfig(
    buy_score=12,
    minimum_adx=18.0,
    volume_multiplier=1.3,
    initial_stop_atr=2.0,
    trailing_stop_atr=2.5,
    trailing_activation_return=0.06,
)

candidate_strategy = replace(
    baseline_strategy,
    initial_stop_atr=1.5,
)


## 2. Baseline ve Candidate dönem karşılaştırması

Holdout burada yalnızca son kontrol için kullanılır. Bu aşamadan sonra holdout dönemi tamamen tüketilmiş kabul edilmelidir.


In [ ]:
strategy_map = {
    "Baseline": baseline_strategy,
    "Candidate_1_5_ATR": candidate_strategy,
}

comparison_records = []
stored_outputs = {}

for strategy_name, strategy_config in strategy_map.items():
    for period_name, (start, end) in PERIODS.items():
        metrics, equity_df, trades_df = evaluate_strategy(
            stock_features=stock_features,
            market_regime=market_regime,
            strategy_config=strategy_config,
            portfolio_config=base_portfolio,
            start=start,
            end=end,
        )

        metrics["Strategy_Name"] = strategy_name
        metrics["Period_Name"] = period_name
        comparison_records.append(metrics)

        stored_outputs[(strategy_name, period_name)] = {
            "equity": equity_df,
            "trades": trades_df,
        }

strategy_period_comparison = pd.DataFrame(comparison_records)

comparison_columns = [
    "Strategy_Name",
    "Period_Name",
    "CAGR_%",
    "Max_Drawdown_%",
    "Profit_Factor",
    "Win_Rate_%",
    "Sharpe",
    "Calmar",
    "Trade_Count",
]

display(
    strategy_period_comparison[comparison_columns]
    .sort_values(["Period_Name", "Strategy_Name"])
)


## 3. Candidate'ın baseline'a göre farkı


In [ ]:
period_pivot = strategy_period_comparison.pivot_table(
    index="Period_Name",
    columns="Strategy_Name",
    values=[
        "CAGR_%",
        "Max_Drawdown_%",
        "Profit_Factor",
        "Sharpe",
        "Calmar",
        "Trade_Count",
    ],
    aggfunc="first",
)

difference_rows = []

for period_name in period_pivot.index:
    difference_rows.append({
        "Period_Name": period_name,
        "CAGR_Difference_pp": (
            period_pivot.loc[
                period_name,
                ("CAGR_%", "Candidate_1_5_ATR"),
            ]
            - period_pivot.loc[
                period_name,
                ("CAGR_%", "Baseline"),
            ]
        ),
        "Drawdown_Difference_pp": (
            period_pivot.loc[
                period_name,
                ("Max_Drawdown_%", "Candidate_1_5_ATR"),
            ]
            - period_pivot.loc[
                period_name,
                ("Max_Drawdown_%", "Baseline"),
            ]
        ),
        "Profit_Factor_Difference": (
            period_pivot.loc[
                period_name,
                ("Profit_Factor", "Candidate_1_5_ATR"),
            ]
            - period_pivot.loc[
                period_name,
                ("Profit_Factor", "Baseline"),
            ]
        ),
        "Sharpe_Difference": (
            period_pivot.loc[
                period_name,
                ("Sharpe", "Candidate_1_5_ATR"),
            ]
            - period_pivot.loc[
                period_name,
                ("Sharpe", "Baseline"),
            ]
        ),
        "Calmar_Difference": (
            period_pivot.loc[
                period_name,
                ("Calmar", "Candidate_1_5_ATR"),
            ]
            - period_pivot.loc[
                period_name,
                ("Calmar", "Baseline"),
            ]
        ),
        "Extra_Trades": (
            period_pivot.loc[
                period_name,
                ("Trade_Count", "Candidate_1_5_ATR"),
            ]
            - period_pivot.loc[
                period_name,
                ("Trade_Count", "Baseline"),
            ]
        ),
    })

strategy_difference = pd.DataFrame(difference_rows)

display(strategy_difference)


## 4. Yıllık performans karşılaştırması


In [ ]:
yearly_frames = []

for strategy_name in strategy_map:
    full_equity = stored_outputs[(strategy_name, "Full")]["equity"]
    yearly = yearly_performance(full_equity)
    yearly["Strategy_Name"] = strategy_name
    yearly_frames.append(yearly)

yearly_comparison = pd.concat(
    yearly_frames,
    ignore_index=True,
)

display(
    yearly_comparison.sort_values(
        ["Year", "Strategy_Name"]
    )
)


In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=yearly_comparison,
    x="Year",
    y="Return_%",
    hue="Strategy_Name",
)

plt.axhline(0, linewidth=1)
plt.title("Baseline ve Candidate — Yıllık Getiriler")
plt.xlabel("Yıl")
plt.ylabel("Getiri (%)")
plt.tight_layout()
plt.show()


## 5. İşlem maliyeti hassasiyeti

Candidate daha fazla işlem ürettiği için daha yüksek maliyetlerde avantajını koruması gerekir.

Senaryolar:

- Normal: %0,20 komisyon + %0,20 slippage
- Yüksek: %0,30 + %0,30
- Stres: %0,40 + %0,40


In [ ]:
cost_scenarios = {
    "Normal_0_2": (0.002, 0.002),
    "High_0_3": (0.003, 0.003),
    "Stress_0_4": (0.004, 0.004),
}

cost_records = []

for strategy_name, strategy_config in strategy_map.items():
    for scenario_name, (
        commission_rate,
        slippage_rate,
    ) in cost_scenarios.items():
        stressed_portfolio = replace(
            base_portfolio,
            commission_rate=commission_rate,
            slippage_rate=slippage_rate,
        )

        for period_name in ["Validation", "Holdout"]:
            start, end = PERIODS[period_name]

            metrics, _, _ = evaluate_strategy(
                stock_features=stock_features,
                market_regime=market_regime,
                strategy_config=strategy_config,
                portfolio_config=stressed_portfolio,
                start=start,
                end=end,
            )

            metrics.update({
                "Strategy_Name": strategy_name,
                "Scenario_Name": scenario_name,
                "Period_Name": period_name,
                "Commission": commission_rate,
                "Slippage": slippage_rate,
            })

            cost_records.append(metrics)

cost_sensitivity = pd.DataFrame(cost_records)

display(
    cost_sensitivity[
        [
            "Strategy_Name",
            "Scenario_Name",
            "Period_Name",
            "CAGR_%",
            "Max_Drawdown_%",
            "Profit_Factor",
            "Sharpe",
            "Calmar",
            "Trade_Count",
        ]
    ].sort_values(
        [
            "Period_Name",
            "Scenario_Name",
            "Strategy_Name",
        ]
    )
)


## 6. Candidate çevresinde parametre platosu

Burada yeni bir kazanan seçmiyoruz. 1.5 ATR sonucunun tek bir keskin noktaya mı bağlı olduğunu, yoksa yakın parametrelerin de benzer performans üretip üretmediğini kontrol ediyoruz.


In [ ]:
neighborhood_grid = {
    "initial_stop_atr": [1.25, 1.50, 1.75],
    "trailing_stop_atr": [2.25, 2.50, 2.75],
    "trailing_activation_return": [0.05, 0.06, 0.07],
}

neighborhood_development = run_strategy_grid(
    stock_features=stock_features,
    market_regime=market_regime,
    base_strategy=baseline_strategy,
    portfolio_config=base_portfolio,
    parameter_grid=neighborhood_grid,
    start=PERIODS["Development"][0],
    end=PERIODS["Development"][1],
)

neighborhood_validation = run_strategy_grid(
    stock_features=stock_features,
    market_regime=market_regime,
    base_strategy=baseline_strategy,
    portfolio_config=base_portfolio,
    parameter_grid=neighborhood_grid,
    start=PERIODS["Validation"][0],
    end=PERIODS["Validation"][1],
)

parameter_columns = [
    "Strategy_initial_stop_atr",
    "Strategy_trailing_stop_atr",
    "Strategy_trailing_activation_return",
]

neighborhood_merged = neighborhood_development.merge(
    neighborhood_validation,
    on=parameter_columns,
    suffixes=("_Development", "_Validation"),
)

neighborhood_merged["Robust_Calmar"] = neighborhood_merged[
    ["Calmar_Development", "Calmar_Validation"]
].min(axis=1)

neighborhood_merged["Robust_PF"] = neighborhood_merged[
    [
        "Profit_Factor_Development",
        "Profit_Factor_Validation",
    ]
].min(axis=1)

neighborhood_merged["CAGR_Gap"] = (
    neighborhood_merged["CAGR_%_Development"]
    - neighborhood_merged["CAGR_%_Validation"]
).abs()

neighborhood_view = neighborhood_merged.sort_values(
    ["Robust_Calmar", "Robust_PF", "CAGR_Gap"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(
    neighborhood_view[
        parameter_columns
        + [
            "CAGR_%_Development",
            "CAGR_%_Validation",
            "Max_Drawdown_%_Development",
            "Max_Drawdown_%_Validation",
            "Profit_Factor_Development",
            "Profit_Factor_Validation",
            "Calmar_Development",
            "Calmar_Validation",
            "Robust_Calmar",
            "Robust_PF",
            "CAGR_Gap",
        ]
    ].head(15)
)


In [ ]:
plateau_summary = pd.DataFrame({
    "Metric": [
        "Median validation Calmar",
        "Minimum validation Profit Factor",
        "Positive validation CAGR ratio",
        "Validation Calmar >= 1 ratio",
    ],
    "Value": [
        neighborhood_view["Calmar_Validation"].median(),
        neighborhood_view[
            "Profit_Factor_Validation"
        ].min(),
        neighborhood_view[
            "CAGR_%_Validation"
        ].gt(0).mean(),
        neighborhood_view[
            "Calmar_Validation"
        ].ge(1.0).mean(),
    ],
})

display(plateau_summary)


## 7. Karar kontrol listesi

Candidate'ın final strateji olması için tercih edilen koşullar:

1. Validation ve Holdout CAGR pozitif.
2. Validation Calmar baseline'ın üzerinde.
3. Validation Profit Factor en az 1,50.
4. Stres maliyet senaryosunda Profit Factor 1'in üzerinde.
5. Yakın parametrelerin çoğu validation döneminde pozitif ve makul sonuç veriyor.
6. Drawdown kötüleşmesi kabul edilebilir seviyede.

Bu koşullar sağlanırsa 1.5 ATR initial stop final sinyal/çıkış konfigürasyonu olarak sabitlenebilir. Sonraki aşama risk yüzdesi ve maksimum pozisyon sayısının portföy seviyesinde incelenmesidir.


In [ ]:
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

strategy_period_comparison.to_csv(
    RESULTS_DIR / "baseline_candidate_period_comparison.csv",
    index=False,
)

strategy_difference.to_csv(
    RESULTS_DIR / "baseline_candidate_differences.csv",
    index=False,
)

yearly_comparison.to_csv(
    RESULTS_DIR / "baseline_candidate_yearly.csv",
    index=False,
)

cost_sensitivity.to_csv(
    RESULTS_DIR / "baseline_candidate_cost_sensitivity.csv",
    index=False,
)

neighborhood_view.to_csv(
    RESULTS_DIR / "candidate_parameter_neighborhood.csv",
    index=False,
)

print("Robustness sonuçları kaydedildi.")
